In [ ]:
# from google.colab import userdata
# KAGGLE_KEY = userdata.get('KAGGLE_KEY')
# KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')

# import os
# import json

# kaggle_dir = os.path.expanduser('~/.kaggle')
# kaggle_json_path = os.path.join(kaggle_dir, 'kaggle.json')

# os.makedirs(kaggle_dir, exist_ok=True)

# kaggle_credentials = {
#     "username": KAGGLE_USERNAME,
#     "key": KAGGLE_KEY
# }

# with open(kaggle_json_path, 'w') as f:
#     json.dump(kaggle_credentials, f)

# os.chmod(kaggle_json_path, 0o600)

# print("kaggle.json created successfully at", kaggle_json_path)

In [ ]:
# ! kaggle competitions download -c triagegeist

In [ ]:
# import zipfile
# import os

# zip_file_path = '/content/triagegeist.zip'
# extract_dir = '/content'

# with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_dir)

# print(f"'{zip_file_path}' unzipped to '{extract_dir}' successfully.")

In [ ]:
import pandas as pd
import os

# List all files in the extraction directory
# directory = '/content'
directory = '/kaggle/input/competitions/triagegeist'
files_in_content = os.listdir(directory)

# Filter for CSV files
csv_files = [f for f in files_in_content if f.endswith('.csv')]

# Dictionary to store DataFrames
dataframes = {}

print(f"Found {len(csv_files)} CSV files: {csv_files}\n")

for csv_file in csv_files:
    df_name = csv_file.replace('.csv', '').replace('-', '_').replace('.', '_') # Create a valid variable name
    file_path = os.path.join(directory, csv_file)
    try:
        df = pd.read_csv(file_path)
        dataframes[df_name] = df
        print(f"Loaded '{csv_file}' into DataFrame '{df_name}'. Showing head:\n")
        display(df.head())
        print("\n" + "-" * 50 + "\n") # Separator for readability
    except Exception as e:
        print(f"Error loading {csv_file}: {e}")

# You can access individual DataFrames like: dataframes['your_csv_file_name_without_extension']

In [ ]:
for df_name, df in dataframes.items():
    print(f"Shape of DataFrame '{df_name}': {df.shape}")

In [ ]:
train_df = dataframes['train']
patient_history_df = dataframes['patient_history']
chief_complaints_df = dataframes['chief_complaints']

# Merge train with patient_history
merged_df = pd.merge(train_df, patient_history_df, on='patient_id', how='left')

# Merge the result with chief_complaints
merged_df = pd.merge(merged_df, chief_complaints_df, on='patient_id', how='left')

print("Shape of merged_df:", merged_df.shape)

In [ ]:
merged_df.info()

In [ ]:
test_df = dataframes['test']

# Merge test with patient_history
merged_test_df = pd.merge(test_df, patient_history_df, on='patient_id', how='left')

# Merge the result with chief_complaints
merged_test_df = pd.merge(merged_test_df, chief_complaints_df, on='patient_id', how='left')

print("Shape of merged_test_df:", merged_test_df.shape)

In [ ]:
merged_df_cols = set(merged_df.columns)
merged_test_df_cols = set(merged_test_df.columns)

# Columns in merged_df but not in merged_test_df
columns_only_in_merged_df = list(merged_df_cols - merged_test_df_cols)
print("Columns in merged_df but not in merged_test_df:")
print(columns_only_in_merged_df)

print("\n" + "-" * 50 + "\n")

# Columns in merged_test_df but not in merged_df
columns_only_in_merged_test_df = list(merged_test_df_cols - merged_df_cols)
print("Columns in merged_test_df but not in merged_df:")
print(columns_only_in_merged_test_df)

In [ ]:
columns_to_drop = ['disposition', 'ed_los_hours', 'triage_nurse_id']

# Drop the specified columns from merged_df
merged_df_processed = merged_df.drop(columns=columns_to_drop)

print("Shape of merged_df after dropping columns:", merged_df_processed.shape)

In [ ]:
merged_df[['chief_complaint_raw', 'chief_complaint_system_x', 'chief_complaint_system_y']]

In [ ]:
(merged_df['chief_complaint_system_x'] == merged_df['chief_complaint_system_y']).value_counts()

In [ ]:
# Drop 'chief_complaint_system_y'
merged_df_processed = merged_df_processed.drop(columns=['chief_complaint_system_y'])

# Rename 'chief_complaint_system_x' to 'chief_complaint_system'
merged_df_processed = merged_df_processed.rename(columns={'chief_complaint_system_x': 'chief_complaint_system'})

print("Shape of merged_df_processed after modifications:", merged_df_processed.shape)

In [ ]:
merged_df_processed.info()

In [ ]:
y_train = merged_df_processed['triage_acuity']
X_train = merged_df_processed.drop(columns=['triage_acuity'])

print("Shape of X_train (features):", X_train.shape)
print("Shape of y_train (target):", y_train.shape)

In [ ]:
columns_to_remove = ['patient_id', 'site_id']
X_train_processed = X_train.drop(columns=columns_to_remove)

print("Shape of X_train after removing patient_id and site_id:", X_train_processed.shape)

In [ ]:
# Identify categorical columns
categorical_cols = X_train_processed.select_dtypes(include=['object']).columns

# Apply one-hot encoding, dropping the first category to avoid multicollinearity
X_train_encoded = pd.get_dummies(X_train_processed, columns=categorical_cols, drop_first=True)

print("Shape of X_train after one-hot encoding:", X_train_encoded.shape)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Split the data into training and validation sets
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_encoded,
    y_train,
    test_size=0.2, # Use 20% of data for validation
    random_state=42,
    stratify=y_train # Stratify to maintain class distribution in splits
)

# Adjust target labels to be 0-indexed for XGBoost
y_train_split_adjusted = y_train_split - 1
y_val_split_adjusted = y_val_split - 1

print("Shape of X_train_split:", X_train_split.shape)
print("Shape of X_val_split:", X_val_split.shape)
print("Shape of y_train_split (adjusted):", y_train_split_adjusted.shape)
print("Shape of y_val_split (adjusted):", y_val_split_adjusted.shape)

# Initialize the XGBoost Classifier
# Use 'multi:softmax' for multi-class classification
# objective='multi:softprob' would output probabilities for each class
# num_class should be the number of unique classes in y_train, which is 5 (1 to 5)
model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(y_train.unique()),
    eval_metric='mlogloss',
    use_label_encoder=False, # Suppress the warning, recommended for future versions
    random_state=42
)

# Train the model
print("\nTraining XGBoost model...")
model.fit(X_train_split, y_train_split_adjusted)
print("Model training complete.")

# Make predictions on the validation set
y_pred = model.predict(X_val_split)

# Evaluate the model
accuracy = accuracy_score(y_val_split_adjusted, y_pred)
print(f"\nAccuracy on validation set: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_val_split_adjusted, y_pred))

In [ ]:
columns_to_remove_test = ['patient_id', 'site_id', 'triage_nurse_id']
merged_test_df_processed = merged_test_df.drop(columns=columns_to_remove_test)

print("Shape of merged_test_df_processed after removing columns:", merged_test_df_processed.shape)

In [ ]:
merged_test_df_processed = merged_test_df_processed.drop(columns=['chief_complaint_system_y'])
merged_test_df_processed = merged_test_df_processed.rename(columns={'chief_complaint_system_x': 'chief_complaint_system'})

print("Shape of merged_test_df_processed after modifications:", merged_test_df_processed.shape)

In [ ]:
merged_df_processed_cols = set(merged_df_processed.columns)
merged_test_df_processed_cols = set(merged_test_df_processed.columns)

# Columns in merged_df_processed but not in merged_test_df_processed
columns_only_in_merged_df_processed = list(merged_df_processed_cols - merged_test_df_processed_cols)
print("Columns in merged_df_processed but not in merged_test_df_processed:")
print(columns_only_in_merged_df_processed)

print("\n" + "-" * 50 + "\n")

# Columns in merged_test_df_processed but not in merged_df_processed
columns_only_in_merged_test_df_processed = list(merged_test_df_processed_cols - merged_df_processed_cols)
print("Columns in merged_test_df_processed but not in merged_df_processed:")
print(columns_only_in_merged_test_df_processed)

In [ ]:
# Identify categorical columns in the processed test DataFrame
categorical_cols_test = merged_test_df_processed.select_dtypes(include=['object']).columns

# Apply one-hot encoding, dropping the first category to avoid multicollinearity
X_test_encoded = pd.get_dummies(merged_test_df_processed, columns=categorical_cols_test, drop_first=True)

# Ensure that the test set has the same columns as the training set
# This is crucial because some categories might only appear in train or test, but not both
missing_cols = set(X_train_encoded.columns) - set(X_test_encoded.columns)
for c in missing_cols:
    X_test_encoded[c] = 0

# Reindex the test DataFrame to have the same columns as the training DataFrame
# This also ensures the order of columns is the same
X_test_encoded = X_test_encoded[X_train_encoded.columns]

print("Shape of X_test after one-hot encoding and alignment:", X_test_encoded.shape)

In [ ]:
# Make predictions on the processed test set
test_predictions_0indexed = model.predict(X_test_encoded)

# Adjust predictions back to 1-indexed (original scale)
test_predictions = test_predictions_0indexed + 1

print("First 10 predictions for triage_acuity on the test set:")
print(test_predictions[:10])
print(f"Number of predictions: {len(test_predictions)}")

In [ ]:
(dataframes['sample_submission'].patient_id == test_df.patient_id).value_counts()

In [ ]:
submission_df = pd.DataFrame({
    'patient_id': test_df['patient_id'],
    'triage_acuity': test_predictions
})

print("Submission DataFrame head:")
display(submission_df.head())
print(f"Submission DataFrame shape: {submission_df.shape}")